# 02 — Construction de la table d'apprentissage

## Ce qu'on va faire ici
On part de ~725 000 lignes de transactions et on les transforme en un seul 
tableau propre où **1 ligne = 1 client**. Ce tableau aura :
- Une vingtaine de **colonnes** qui décrivent chaque client (ses features)
- Une colonne **CHURN** qui indique si le client a décroché ou pas

C'est ce tableau qui servira pour la segmentation (étape 3) et la 
prédiction de churn (étape 4).

## La règle d'or : le split temporel

On coupe le temps en deux autour de notre date de référence **T0 = 1er juin 2011** :

- **Avant T0** → on regarde le passé pour construire les caractéristiques 
  du client (combien il a commandé, combien il a dépensé, etc.)
- **Après T0** → on regarde si le client est revenu dans les 6 mois 
  suivants. S'il n'est pas revenu, on le marque comme "churné".

Cette séparation est essentielle. Si on mélangeait, le modèle utiliserait 
des informations du futur pour prédire le futur. C'est l'erreur classique 
qu'on évite à tout prix en machine learning.

## Les 4 familles de features qu'on va construire

1. **RFM classique** : Recency, Frequency, Monetary — le standard du 
   marketing depuis 30 ans
2. **Comportement d'achat** : ticket moyen, diversité produit, régularité
3. **Comportement temporel** : ancienneté, intervalle moyen entre commandes
4. **Comportement d'insatisfaction** : taux d'annulation (signal rare et 
   précieux qu'on retrouve peu dans les projets classiques)

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Affichage propre
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

# Connexion à la base DuckDB de l'étape 1
con = duckdb.connect("../data/retail.duckdb")

# === Paramètres temporels ===
T0 = "2011-06-01"                  # Date de coupure
PREDICTION_WINDOW_DAYS = 180       # 6 mois pour observer le retour
OBSERVATION_START = "2009-12-01"   # Début des données exploitables

# Dossier de sortie
DATA_PROCESSED = Path("../data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"✅ Connexion à la base : ../data/retail.duckdb")
print(f"✅ T0 (date de coupure) : {T0}")
print(f"✅ Fenêtre de prédiction : {PREDICTION_WINDOW_DAYS} jours")

✅ Connexion à la base : ../data/retail.duckdb
✅ T0 (date de coupure) : 2011-06-01
✅ Fenêtre de prédiction : 180 jours


## 2. Définir la cohorte de clients à étudier

Un client est éligible s'il a commandé **au moins une fois avant T0**. Si 
on n'a aucune transaction avant T0, on n'a aucune information sur lui 
pour faire une prédiction — autant l'exclure.

On va probablement perdre un peu de monde par rapport aux 5 350 clients UK 
identifiés à l'étape 1, car certains n'ont commandé qu'après juin 2011. 
C'est normal et attendu.

In [4]:
n_cohorte = con.execute(f"""
    SELECT COUNT(DISTINCT "Customer ID") AS n_clients
    FROM transactions_clean
    WHERE InvoiceDate < '{T0}'
""").fetchone()[0]

print(f"Cohorte éligible : {n_cohorte:,} clients")
print(f"   (clients ayant commandé au moins une fois avant le {T0})")

Cohorte éligible : 4,518 clients
   (clients ayant commandé au moins une fois avant le 2011-06-01)


## 3. Construire la cible CHURN

Pour chaque client de la cohorte, on regarde s'il a passé au moins une 
commande entre le 1er juin 2011 et le 28 novembre 2011 (180 jours après T0).

**Règle simple** :
- `churn = 1` → le client n'a passé aucune commande pendant cette période → il a décroché
- `churn = 0` → le client a passé au moins une commande → il est resté actif

Cette définition est défendable : elle correspond à la pratique standard 
du marketing pour les retailers non-abonnement.

### Ce qu'on attend en sortie
Vu la nature B2B/cadeaux du retailer, on s'attend à un taux de churn 
autour de **35-50%**. C'est l'idéal pour entraîner un modèle ML — ni 
trop déséquilibré, ni trop équilibré.

In [5]:
# Construction de la table target
con.execute(f"""
    CREATE OR REPLACE TABLE customer_target AS
    WITH cohorte AS (
        -- Tous les clients ayant commandé avant T0
        SELECT DISTINCT "Customer ID" AS customer_id
        FROM transactions_clean
        WHERE InvoiceDate < '{T0}'
    ),
    orders_after_t0 AS (
        -- Commandes passées dans la fenêtre de prédiction
        SELECT 
            "Customer ID" AS customer_id,
            COUNT(DISTINCT Invoice) AS n_orders_future
        FROM transactions_clean
        WHERE InvoiceDate >= '{T0}'
          AND InvoiceDate < DATE_ADD(DATE '{T0}', INTERVAL {PREDICTION_WINDOW_DAYS} DAY)
        GROUP BY 1
    )
    SELECT 
        c.customer_id,
        COALESCE(o.n_orders_future, 0) AS n_orders_future,
        CASE WHEN COALESCE(o.n_orders_future, 0) = 0 THEN 1 ELSE 0 END AS churn
    FROM cohorte c
    LEFT JOIN orders_after_t0 o ON c.customer_id = o.customer_id
""")

# Vérification de la distribution
target_dist = con.execute("""
    SELECT 
        churn,
        COUNT(*) AS n_customers,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM customer_target
    GROUP BY churn
    ORDER BY churn
""").df()

print("=== Distribution de la cible CHURN ===")
print(target_dist)

# Verdict
churn_rate = target_dist[target_dist['churn'] == 1]['pct'].values[0]
print(f"\nTaux de churn : {churn_rate}%")
if 25 <= churn_rate <= 60:
    print("✅ Taux de churn dans la zone idéale (25-60%) pour le ML")
elif churn_rate > 60:
    print("⚠️  Déséquilibre en faveur du churn — on devra utiliser class_weight")
else:
    print("⚠️  Peu de churn — à analyser")

=== Distribution de la cible CHURN ===
   churn  n_customers    pct
0      0         2308  51.08
1      1         2210  48.92

Taux de churn : 48.92%
✅ Taux de churn dans la zone idéale (25-60%) pour le ML


## 4. Famille 1 — Les features RFM classiques

On calcule les 3 indicateurs historiques du marketing client :

- **Recency** : depuis combien de jours le client n'a-t-il pas commandé ? 
  (calculé par rapport à T0, pas à aujourd'hui)
- **Frequency** : combien de commandes distinctes a-t-il passées ?
- **Monetary** : combien a-t-il dépensé au total (en livres sterling) ?

### Astuce technique
Pour calculer le Monetary, on utilise `Quantity × Price` (le dataset ne 
contient pas de colonne "total"). On somme ensuite tout par client.

### Important
Toutes ces features sont calculées **uniquement sur les données avant T0**. 
C'est la règle du split temporel : aucune information du futur ne doit 
remonter dans les features.

In [13]:
con.execute(f"""
    CREATE OR REPLACE TABLE features_rfm AS
    SELECT 
        "Customer ID" AS customer_id,
        DATE_DIFF('day', MAX(InvoiceDate), DATE '{T0}') AS recency,
        COUNT(DISTINCT Invoice) AS frequency,
        ROUND(SUM(Quantity * Price), 2) AS monetary
    FROM transactions_clean
    WHERE InvoiceDate < '{T0}'
    GROUP BY "Customer ID"
""")

# Aperçu et stats descriptives
rfm_df = con.execute("SELECT * FROM features_rfm").df()
print(f"Features RFM calculées sur {len(rfm_df):,} clients\n")
print(rfm_df[['recency', 'frequency', 'monetary']].describe())

Features RFM calculées sur 4,518 clients

           recency    frequency       monetary
count  4518.000000  4518.000000    4518.000000
mean    170.676848     5.222222    2244.265170
std     142.109212     9.482012    8853.131173
min       1.000000     1.000000       2.900000
25%      44.000000     1.000000     310.705000
50%     167.000000     3.000000     736.400000
75%     244.000000     6.000000    1934.402500
max     547.000000   213.000000  413806.460000


## 5. Famille 2 — Le comportement d'achat

Le RFM dit "combien" et "quand", mais pas "comment". On enrichit avec :

- **Ticket moyen** : prix moyen d'une commande. Permet de distinguer un 
  petit client régulier d'un gros client occasionnel.
- **Écart-type des tickets** : un client régulier a peu de variance, un 
  client erratique en a beaucoup. C'est un signal de **prévisibilité**.
- **Nombre de produits distincts** : un client qui explore le catalogue 
  est probablement plus engagé qu'un qui rachète toujours la même chose.

Ces features apportent de la finesse là où le RFM brut serait flou.

In [16]:
con.execute(f"""
    CREATE OR REPLACE TABLE features_behavior AS
    WITH order_totals AS (
        -- Total dépensé par commande
        SELECT 
            "Customer ID" AS customer_id,
            Invoice,
            SUM(Quantity * Price) AS order_total
        FROM transactions_clean
        WHERE InvoiceDate < '{T0}'
        GROUP BY 1, 2
    ),
    product_diversity AS (
        -- Nombre de produits distincts par client
        SELECT 
            "Customer ID" AS customer_id,
            COUNT(DISTINCT StockCode) AS n_distinct_products
        FROM transactions_clean
        WHERE InvoiceDate < '{T0}'
        GROUP BY 1
    )
    SELECT 
        ot.customer_id,
        ROUND(AVG(ot.order_total), 2) AS avg_basket,
        ROUND(STDDEV(ot.order_total), 2) AS std_basket,
        pd.n_distinct_products
    FROM order_totals ot
    JOIN product_diversity pd ON ot.customer_id = pd.customer_id
    GROUP BY ot.customer_id, pd.n_distinct_products
""")

behavior_df = con.execute("SELECT * FROM features_behavior").df()
print(f"Features comportementales calculées sur {len(behavior_df):,} clients\n")
print(behavior_df.describe())

Features comportementales calculées sur 4,518 clients

        customer_id    avg_basket   std_basket  n_distinct_products
count   4518.000000   4518.000000   3105.00000          4518.000000
mean   15564.579903    353.257430    205.21372            69.697875
std     1577.380408    478.991878    519.91045            93.479151
min    12346.000000      2.900000      0.00000             1.000000
25%    14212.250000    176.847500     68.51000            17.000000
50%    15578.000000    279.860000    129.74000            39.000000
75%    16932.750000    407.955000    223.14000            86.000000
max    18287.000000  14844.770000  22271.23000          1764.000000


## 6. Famille 3 — Le comportement temporel

On va au-delà des chiffres bruts pour capturer le **rythme** du client :

- **Ancienneté** (`tenure`) : depuis combien de jours est-il client ? 
  (date de sa 1ère commande jusqu'à T0)
- **Intervalle moyen entre commandes** (`avg_interpurchase`) : son rythme 
  d'achat habituel. Un client qui commande tous les 15 jours est différent 
  d'un qui commande tous les 90 jours.

### Pourquoi ces features sont précieuses
Elles capturent la **dynamique** du client, pas juste son état actuel. 
Deux clients avec les mêmes Recency/Frequency peuvent avoir des rythmes 
très différents — l'un commande régulièrement, l'autre par à-coups.

In [17]:
con.execute(f"""
    CREATE OR REPLACE TABLE features_temporal AS
    WITH order_dates AS (
        -- Une ligne par commande avec sa date
        SELECT 
            "Customer ID" AS customer_id,
            Invoice,
            MIN(InvoiceDate) AS order_date
        FROM transactions_clean
        WHERE InvoiceDate < '{T0}'
        GROUP BY 1, 2
    ),
    temporal_stats AS (
        SELECT 
            customer_id,
            DATE_DIFF('day', MIN(order_date), DATE '{T0}') AS tenure_days,
            CASE 
                WHEN COUNT(*) > 1 THEN 
                    DATE_DIFF('day', MIN(order_date), MAX(order_date)) * 1.0 / (COUNT(*) - 1)
                ELSE NULL
            END AS avg_interpurchase_days
        FROM order_dates
        GROUP BY customer_id
    )
    SELECT * FROM temporal_stats
""")

temporal_df = con.execute("SELECT * FROM features_temporal").df()
print(f"✅ Features temporelles calculées sur {len(temporal_df):,} clients\n")
print(temporal_df.describe())

✅ Features temporelles calculées sur 4,518 clients

        customer_id  tenure_days  avg_interpurchase_days
count   4518.000000  4518.000000             3105.000000
mean   15564.579903   360.670872               83.450475
std     1577.380408   153.995320               73.927728
min    12346.000000     1.000000                0.000000
25%    14212.250000   234.000000               35.125000
50%    15578.000000   398.000000               62.166667
75%    16932.750000   493.000000              106.500000
max    18287.000000   547.000000              540.000000


## 7. Famille 4 — Le comportement d'insatisfaction

À l'étape 1, on a vu que 
le dataset contient 19 494 annulations (les factures qui commencent par 
'C').

### Hypothèse business
Un client qui annule beaucoup de commandes est probablement insatisfait → 
candidat sérieux au churn.

### Features construites
- **Nombre d'annulations** avant T0
- **Taux d'annulation** = annulations / total commandes

### Note méthodologique
Pour cette famille uniquement, on requête `transactions_raw` (et pas 
`transactions_clean`) parce que les annulations ont été filtrées dans 
la table clean. On veut justement les compter ici.

In [19]:
con.execute(f"""
    CREATE OR REPLACE TABLE features_cancellations AS
    WITH cancellations AS (
        SELECT 
            "Customer ID" AS customer_id,
            COUNT(DISTINCT Invoice) AS n_cancellations
        FROM transactions_raw
        WHERE InvoiceDate < '{T0}'
          AND "Customer ID" IS NOT NULL
          AND Country = 'United Kingdom'
          AND Invoice LIKE 'C%'
        GROUP BY 1
    ),
    total_orders AS (
        SELECT 
            customer_id,
            frequency AS n_orders
        FROM features_rfm
    )
    SELECT 
        t.customer_id,
        COALESCE(c.n_cancellations, 0) AS n_cancellations,
        ROUND(COALESCE(c.n_cancellations, 0) * 1.0 / t.n_orders, 4) AS cancellation_rate
    FROM total_orders t
    LEFT JOIN cancellations c ON t.customer_id = c.customer_id
""")

cancel_df = con.execute("SELECT * FROM features_cancellations").df()
print(f"Features d'annulation calculées sur {len(cancel_df):,} clients\n")
print(cancel_df.describe())
print(f"\nClients ayant au moins 1 annulation : {(cancel_df['n_cancellations'] > 0).sum():,}")
print(f"Soit {(cancel_df['n_cancellations'] > 0).mean()*100:.1f}% de la cohorte")

Features d'annulation calculées sur 4,518 clients

        customer_id  n_cancellations  cancellation_rate
count   4518.000000      4518.000000        4518.000000
mean   15564.579903         1.127711           0.199636
std     1577.380408         2.743209           0.354621
min    12346.000000         0.000000           0.000000
25%    14212.250000         0.000000           0.000000
50%    15578.000000         0.000000           0.000000
75%    16932.750000         1.000000           0.285700
max    18287.000000        52.000000           4.000000

Clients ayant au moins 1 annulation : 1,850
Soit 40.9% de la cohorte
